In [34]:
# ============================================================
# INDIVIDUAL ASSIGNMENT 1 — SEARCH
# MASTER / INSTRUCTOR VERSION
# ============================================================
#
# Required files in the current working directory:
#
#   cities.csv
#   connections.csv
#
# IMPORTANT:
# - No absolute paths.
# - No Google Drive paths.
# - No machine-specific paths.
# - City names use one canonical normalization.
# - Successors are always processed alphabetically.
#
# ============================================================

In [35]:
# ============================================================
# PROVIDED TO STUDENTS
# 1. IMPORTS
# ============================================================

from dataclasses import dataclass
from pathlib import Path
from collections import deque
import heapq
import itertools
import json
import math
import re
import unicodedata

import pandas as pd
from IPython.display import display

In [36]:
# ============================================================
# PROVIDED TO STUDENTS
# 2. GLOBAL CONSTANTS
# ============================================================

CITIES_FILE = Path("Data/cities.csv")
CONNECTIONS_FILE = Path("Data/connections.csv")

EXPECTED_NUMBER_OF_CITIES = 20
EXPECTED_NUMBER_OF_CONNECTIONS = 35

EARTH_RADIUS_KM = 6371.0

EPS = 1e-12

In [37]:

# ============================================================
# PROVIDED TO STUDENTS
# 3. CANONICAL CITY NORMALIZATION
# ============================================================

def normalize_city(city):
    """
    Convert city names to one canonical representation.

    Examples
    --------
    "Bucharest"     -> "bucharest"
    " Iași "        -> "iasi"
    "Iasi"          -> "iasi"
    "Timișoara"     -> "timisoara"

    "Cluj-Napoca"   -> "cluj_napoca"
    "Cluj_Napoca"   -> "cluj_napoca"
    "Cluj Napoca"   -> "cluj_napoca"
    "Cluj–Napoca"   -> "cluj_napoca"

    Do not modify this function.
    """

    city = str(city).strip().lower()

    # Remove accents / diacritics.
    city = unicodedata.normalize(
        "NFKD",
        city,
    )

    city = "".join(
        character
        for character in city
        if not unicodedata.combining(character)
    )

    # Convert spaces, underscores and different dash
    # characters into one canonical separator.
    city = re.sub(
        r"[\s_\-–—−]+",
        "_",
        city,
    )

    city = city.strip("_")

    return city

In [38]:

# ============================================================
# PROVIDED TO STUDENTS
# 4. CHECK FILES
# ============================================================

if not CITIES_FILE.exists():
    raise FileNotFoundError(
        "cities.csv was not found in the current working directory."
    )

if not CONNECTIONS_FILE.exists():
    raise FileNotFoundError(
        "connections.csv was not found in the current working directory."
    )

In [39]:

# ============================================================
# PROVIDED TO STUDENTS
# 5. LOAD DATA
# ============================================================

cities_df = pd.read_csv(
    CITIES_FILE
)

connections_df = pd.read_csv(
    CONNECTIONS_FILE
)

In [40]:

# ============================================================
# PROVIDED TO STUDENTS
# 6. VALIDATE REQUIRED COLUMNS
# ============================================================

required_city_columns = {
    "city",
    "latitude",
    "longitude",
}

required_connection_columns = {
    "city_from",
    "city_to",
    "distance_km",
    "time_min",
}


missing_city_columns = (
    required_city_columns
    - set(cities_df.columns)
)

missing_connection_columns = (
    required_connection_columns
    - set(connections_df.columns)
)


if missing_city_columns:
    raise ValueError(
        "cities.csv is missing required columns: "
        f"{sorted(missing_city_columns)}"
    )

if missing_connection_columns:
    raise ValueError(
        "connections.csv is missing required columns: "
        f"{sorted(missing_connection_columns)}"
    )


In [41]:
# ============================================================
# PROVIDED TO STUDENTS
# 3. CANONICAL CITY NORMALIZATION
# ============================================================

def normalize_city(city):
    """
    Convert city names to one canonical representation.

    Examples
    --------
    "Bucharest"     -> "bucharest"
    " Iași "        -> "iasi"
    "Iasi"          -> "iasi"
    "Timișoara"     -> "timisoara"

    "Cluj-Napoca"   -> "cluj_napoca"
    "Cluj_Napoca"   -> "cluj_napoca"
    "Cluj Napoca"   -> "cluj_napoca"
    "Cluj–Napoca"   -> "cluj_napoca"

    Do not modify this function.
    """

    city = str(city).strip().lower()

    # Remove accents / diacritics.
    city = unicodedata.normalize(
        "NFKD",
        city,
    )

    city = "".join(
        character
        for character in city
        if not unicodedata.combining(character)
    )

    # Convert spaces, underscores and different dash
    # characters into one canonical separator.
    city = re.sub(
        r"[\s_\-–—−]+",
        "_",
        city,
    )

    city = city.strip("_")

    return city



In [42]:

# ============================================================
# PROVIDED TO STUDENTS
# 8. CONVERT NUMERIC COLUMNS
# ============================================================

cities_df["latitude"] = (
    cities_df["latitude"]
    .astype(float)
)

cities_df["longitude"] = (
    cities_df["longitude"]
    .astype(float)
)

connections_df["distance_km"] = (
    connections_df["distance_km"]
    .astype(float)
)

connections_df["time_min"] = (
    connections_df["time_min"]
    .astype(float)
)


In [43]:

# ============================================================
# PROVIDED TO STUDENTS
# 9. DATASET INTEGRITY CHECKS
# ============================================================

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

assert not cities_df[
    [
        "city",
        "latitude",
        "longitude",
    ]
].isna().any().any(), (
    "cities.csv contains missing values."
)


assert not connections_df[
    [
        "city_from",
        "city_to",
        "distance_km",
        "time_min",
    ]
].isna().any().any(), (
    "connections.csv contains missing values."
)

In [44]:

# ------------------------------------------------------------
# Positive costs
# ------------------------------------------------------------

assert (
    connections_df["distance_km"] > 0
).all(), (
    "All distances must be positive."
)


assert (
    connections_df["time_min"] > 0
).all(), (
    "All travel times must be positive."
)

In [45]:

# ------------------------------------------------------------
# Duplicate canonical city names
# ------------------------------------------------------------

duplicated_cities = (
    cities_df[
        cities_df["city"].duplicated(
            keep=False
        )
    ]["city"]
    .unique()
    .tolist()
)


if duplicated_cities:
    raise ValueError(
        "cities.csv contains duplicate cities "
        "after normalization: "
        f"{sorted(duplicated_cities)}"
    )

In [46]:

# ------------------------------------------------------------
# Every connection city must exist in cities.csv
# ------------------------------------------------------------

known_cities = set(
    cities_df["city"]
)

connection_cities = (
    set(connections_df["city_from"])
    |
    set(connections_df["city_to"])
)

unknown_connection_cities = (
    connection_cities
    - known_cities
)


if unknown_connection_cities:
    raise ValueError(
        "connections.csv contains cities that do not "
        "exist in cities.csv after normalization: "
        f"{sorted(unknown_connection_cities)}"
    )

ValueError: connections.csv contains cities that do not exist in cities.csv after normalization: ['athens', 'bari', 'belgrade', 'bratislava', 'bucharest', 'budapest', 'cluj_napoca', 'iasi', 'ljubljana', 'naples', 'plovdiv', 'rome', 'sarajevo', 'skopje', 'sofia', 'thessaloniki', 'timisoara', 'varna', 'vienna', 'zagreb']

In [ ]:

# ------------------------------------------------------------
# No self-connections
# ------------------------------------------------------------

self_connections = connections_df[
    connections_df["city_from"]
    ==
    connections_df["city_to"]
]


assert self_connections.empty, (
    "The dataset contains self-connections."
)


In [ ]:

# ------------------------------------------------------------
# No duplicate undirected connections
# ------------------------------------------------------------

connection_pairs = connections_df.apply(
    lambda row: tuple(
        sorted(
            (
                row["city_from"],
                row["city_to"],
            )
        )
    ),
    axis=1,
)


duplicate_connections = (
    connection_pairs[
        connection_pairs.duplicated(
            keep=False
        )
    ]
    .unique()
    .tolist()
)


assert not duplicate_connections, (
    "Duplicate undirected connections found: "
    f"{duplicate_connections}"
)

In [ ]:
# ============================================================
# INDIVIDUAL ASSIGNMENT 1 — SEARCH
# MASTER / INSTRUCTOR VERSION
# ============================================================
#
# Required files in the current working directory:
#
#   cities.csv
#   connections.csv
#
# IMPORTANT:
# - No absolute paths.
# - No Google Drive paths.
# - No machine-specific paths.
# - City names use one canonical normalization.
# - Successors are always processed alphabetically.
#
# ============================================================


# ============================================================
# PROVIDED TO STUDENTS
# 1. IMPORTS
# ============================================================

from dataclasses import dataclass
from pathlib import Path
from collections import deque
import heapq
import itertools
import json
import math
import re
import unicodedata

import pandas as pd
from IPython.display import display


# ============================================================
# PROVIDED TO STUDENTS
# 2. GLOBAL CONSTANTS
# ============================================================

CITIES_FILE = Path("Data/cities.csv")
CONNECTIONS_FILE = Path("Data/connections.csv")

EXPECTED_NUMBER_OF_CITIES = 20
EXPECTED_NUMBER_OF_CONNECTIONS = 35

EARTH_RADIUS_KM = 6371.0

EPS = 1e-12


# ============================================================
# PROVIDED TO STUDENTS
# 3. CANONICAL CITY NORMALIZATION
# ============================================================

def normalize_city(city):
    """
    Convert city names to one canonical representation.

    Examples
    --------
    "Bucharest"     -> "bucharest"
    " Iași "        -> "iasi"
    "Iasi"          -> "iasi"
    "Timișoara"     -> "timisoara"

    "Cluj-Napoca"   -> "cluj_napoca"
    "Cluj_Napoca"   -> "cluj_napoca"
    "Cluj Napoca"   -> "cluj_napoca"
    "Cluj–Napoca"   -> "cluj_napoca"

    Do not modify this function.
    """

    city = str(city).strip().lower()

    # Remove accents / diacritics.
    city = unicodedata.normalize(
        "NFKD",
        city,
    )

    city = "".join(
        character
        for character in city
        if not unicodedata.combining(character)
    )

    # Convert spaces, underscores and different dash
    # characters into one canonical separator.
    city = re.sub(
        r"[\s_\-–—−]+",
        "_",
        city,
    )

    city = city.strip("_")

    return city


# ============================================================
# PROVIDED TO STUDENTS
# 4. CHECK FILES
# ============================================================

if not CITIES_FILE.exists():
    raise FileNotFoundError(
        "cities.csv was not found in the current working directory."
    )

if not CONNECTIONS_FILE.exists():
    raise FileNotFoundError(
        "connections.csv was not found in the current working directory."
    )


# ============================================================
# PROVIDED TO STUDENTS
# 5. LOAD DATA
# ============================================================

cities_df = pd.read_csv(
    CITIES_FILE
)

connections_df = pd.read_csv(
    CONNECTIONS_FILE
)


# ============================================================
# PROVIDED TO STUDENTS
# 6. VALIDATE REQUIRED COLUMNS
# ============================================================

required_city_columns = {
    "city",
    "latitude",
    "longitude",
}

required_connection_columns = {
    "city_from",
    "city_to",
    "distance_km",
    "time_min",
}


missing_city_columns = (
    required_city_columns
    - set(cities_df.columns)
)

missing_connection_columns = (
    required_connection_columns
    - set(connections_df.columns)
)


if missing_city_columns:
    raise ValueError(
        "cities.csv is missing required columns: "
        f"{sorted(missing_city_columns)}"
    )

if missing_connection_columns:
    raise ValueError(
        "connections.csv is missing required columns: "
        f"{sorted(missing_connection_columns)}"
    )


# ============================================================
# PROVIDED TO STUDENTS
# 7. NORMALIZE CITY NAMES
# ============================================================

cities_df["city"] = (
    cities_df["city"]
    .map(normalize_city)
)

connections_df["city_from"] = (
    connections_df["city_from"]
    .map(normalize_city)
)

connections_df["city_to"] = (
    connections_df["city_to"]
    .map(normalize_city)
)


# ============================================================
# PROVIDED TO STUDENTS
# 8. CONVERT NUMERIC COLUMNS
# ============================================================

cities_df["latitude"] = (
    cities_df["latitude"]
    .astype(float)
)

cities_df["longitude"] = (
    cities_df["longitude"]
    .astype(float)
)

connections_df["distance_km"] = (
    connections_df["distance_km"]
    .astype(float)
)

connections_df["time_min"] = (
    connections_df["time_min"]
    .astype(float)
)


# ============================================================
# PROVIDED TO STUDENTS
# 9. DATASET INTEGRITY CHECKS
# ============================================================

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

assert not cities_df[
    [
        "city",
        "latitude",
        "longitude",
    ]
].isna().any().any(), (
    "cities.csv contains missing values."
)


assert not connections_df[
    [
        "city_from",
        "city_to",
        "distance_km",
        "time_min",
    ]
].isna().any().any(), (
    "connections.csv contains missing values."
)


# ------------------------------------------------------------
# Positive costs
# ------------------------------------------------------------

assert (
    connections_df["distance_km"] > 0
).all(), (
    "All distances must be positive."
)


assert (
    connections_df["time_min"] > 0
).all(), (
    "All travel times must be positive."
)


# ------------------------------------------------------------
# Duplicate canonical city names
# ------------------------------------------------------------

duplicated_cities = (
    cities_df[
        cities_df["city"].duplicated(
            keep=False
        )
    ]["city"]
    .unique()
    .tolist()
)


if duplicated_cities:
    raise ValueError(
        "cities.csv contains duplicate cities "
        "after normalization: "
        f"{sorted(duplicated_cities)}"
    )


# ------------------------------------------------------------
# Every connection city must exist in cities.csv
# ------------------------------------------------------------

known_cities = set(
    cities_df["city"]
)

connection_cities = (
    set(connections_df["city_from"])
    |
    set(connections_df["city_to"])
)

unknown_connection_cities = (
    connection_cities
    - known_cities
)


if unknown_connection_cities:
    raise ValueError(
        "connections.csv contains cities that do not "
        "exist in cities.csv after normalization: "
        f"{sorted(unknown_connection_cities)}"
    )


# ------------------------------------------------------------
# No self-connections
# ------------------------------------------------------------

self_connections = connections_df[
    connections_df["city_from"]
    ==
    connections_df["city_to"]
]


assert self_connections.empty, (
    "The dataset contains self-connections."
)


# ------------------------------------------------------------
# No duplicate undirected connections
# ------------------------------------------------------------

connection_pairs = connections_df.apply(
    lambda row: tuple(
        sorted(
            (
                row["city_from"],
                row["city_to"],
            )
        )
    ),
    axis=1,
)


duplicate_connections = (
    connection_pairs[
        connection_pairs.duplicated(
            keep=False
        )
    ]
    .unique()
    .tolist()
)


assert not duplicate_connections, (
    "Duplicate undirected connections found: "
    f"{duplicate_connections}"
)


# ------------------------------------------------------------
# Expected dataset size
# ------------------------------------------------------------

assert len(cities_df) == EXPECTED_NUMBER_OF_CITIES, (
    "Unexpected number of cities. "
    f"Expected {EXPECTED_NUMBER_OF_CITIES}, "
    f"found {len(cities_df)}."
)


assert len(connections_df) == EXPECTED_NUMBER_OF_CONNECTIONS, (
    "Unexpected number of connections. "
    f"Expected {EXPECTED_NUMBER_OF_CONNECTIONS}, "
    f"found {len(connections_df)}."
)


# ============================================================
# PROVIDED TO STUDENTS
# 10. BUILD BIDIRECTIONAL GRAPH
# ============================================================

graph = {
    city: {}
    for city in cities_df["city"]
}


for row in connections_df.itertuples(
    index=False
):

    city_a = row.city_from
    city_b = row.city_to

    edge_data = {
        "distance_km":
            float(row.distance_km),

        "time_min":
            float(row.time_min),
    }

    graph[city_a][city_b] = (
        edge_data.copy()
    )

    graph[city_b][city_a] = (
        edge_data.copy()
    )


# ============================================================
# PROVIDED TO STUDENTS
# 11. CITY COORDINATES
# ============================================================

positions = {
    row.city: (
        float(row.latitude),
        float(row.longitude),
    )
    for row in cities_df.itertuples(
        index=False
    )
}


# ============================================================
# PROVIDED TO STUDENTS
# 12. GRAPH VALIDATION
# ============================================================

# ------------------------------------------------------------
# Check bidirectionality
# ------------------------------------------------------------

for city, neighbours in graph.items():

    for neighbour in neighbours:

        assert city in graph[neighbour], (
            "Graph is not bidirectional: "
            f"{city} -> {neighbour}"
        )

        assert math.isclose(
            graph[city][neighbour]["distance_km"],
            graph[neighbour][city]["distance_km"],
        )

        assert math.isclose(
            graph[city][neighbour]["time_min"],
            graph[neighbour][city]["time_min"],
        )


# ------------------------------------------------------------
# Check graph connectivity
# ------------------------------------------------------------

first_city = next(
    iter(graph)
)

visited = {
    first_city
}

stack = [
    first_city
]


while stack:

    current = stack.pop()

    for neighbour in graph[current]:

        if neighbour not in visited:

            visited.add(
                neighbour
            )

            stack.append(
                neighbour
            )


assert len(visited) == len(graph), (
    "The graph is not connected."
)


print(
    "Number of cities:",
    len(graph)
)

print(
    "Number of connections:",
    len(connections_df)
)

print(
    "Dataset loaded successfully."
)


print(
    "\nCanonical city identifiers:"
)

for city in sorted(graph):

    print(
        f"  - {city}"
    )


# ============================================================
# PROVIDED TO STUDENTS
# 13. SEARCH NODE
# ============================================================

@dataclass
class Node:

    state: str

    parent: "Node | None" = None

    path_cost: float = 0.0

    depth: int = 0


# ============================================================
# PROVIDED TO STUDENTS
# 14. SUCCESSORS
# ============================================================

def successors(state):
    """
    Return successor states in alphabetical order.

    Do not modify this function.
    """

    state = normalize_city(
        state
    )

    if state not in graph:
        raise KeyError(
            f"Unknown city: {state}"
        )

    return sorted(
        graph[state].keys()
    )


# ============================================================
# PROVIDED TO STUDENTS
# 15. RECONSTRUCT PATH
# ============================================================

def reconstruct_path(node):

    path = []

    while node is not None:

        path.append(
            node.state
        )

        node = (
            node.parent
        )

    return list(
        reversed(path)
    )


# ============================================================
# PROVIDED TO STUDENTS
# 16. PATH METRICS
# ============================================================

def path_metrics(path):

    if path is None:
        return {
            "path_length": None,
            "distance_km": None,
            "time_min": None,
        }

    distance = 0.0
    time = 0.0

    for city_a, city_b in zip(
        path[:-1],
        path[1:],
    ):

        distance += (
            graph[city_a]
            [city_b]
            ["distance_km"]
        )

        time += (
            graph[city_a]
            [city_b]
            ["time_min"]
        )

    return {
        "path_length":
            len(path) - 1,

        "distance_km":
            float(distance),

        "time_min":
            float(time),
    }


# ============================================================
# PROVIDED TO STUDENTS
# 17. VALIDATE PATH
# ============================================================

def validate_path(
    path,
    start,
    goal,
):

    if not path:
        return False

    start = normalize_city(
        start
    )

    goal = normalize_city(
        goal
    )

    # Require canonical city identifiers.
    if path[0] != start:
        return False

    if path[-1] != goal:
        return False

    for city in path:

        if city not in graph:
            return False

    for city_a, city_b in zip(
        path[:-1],
        path[1:],
    ):

        if city_b not in graph[city_a]:
            return False

    return True


# ============================================================
# PROVIDED TO STUDENTS
# 18. STANDARD RESULT FORMAT
# ============================================================

def build_result(
    algorithm,
    node,
    expanded_nodes,
    generated_nodes,
    max_frontier_size,
    expanded_order,
):

    # --------------------------------------------------------
    # Failure
    # --------------------------------------------------------

    if node is None:

        return {

            "algorithm":
                algorithm,

            "solution": {

                "found":
                    False,

                "path":
                    None,

                "path_length":
                    None,

                "distance_km":
                    None,

                "time_min":
                    None,
            },

            "diagnostics": {

                "expanded_nodes":
                    int(expanded_nodes),

                "generated_nodes":
                    int(generated_nodes),

                "max_frontier_size":
                    int(max_frontier_size),

                "expanded_order":
                    list(expanded_order),
            },
        }

    # --------------------------------------------------------
    # Success
    # --------------------------------------------------------

    path = reconstruct_path(
        node
    )

    metrics = path_metrics(
        path
    )

    return {

        "algorithm":
            algorithm,

        "solution": {

            "found":
                True,

            "path":
                path,

            **metrics,
        },

        "diagnostics": {

            "expanded_nodes":
                int(expanded_nodes),

            "generated_nodes":
                int(generated_nodes),

            "max_frontier_size":
                int(max_frontier_size),

            "expanded_order":
                list(expanded_order),
        },
    }


# ============================================================
# PROVIDED TO STUDENTS
# METRIC DEFINITIONS
# ============================================================
#
# expanded_nodes:
#
#   Number of nodes removed from the frontier for which
#   successors are generated.
#
#   A goal node is not counted as expanded if the search
#   stops immediately after selecting it.
#
#
# generated_nodes:
#
#   Number of child search nodes actually created and added
#   to the frontier.
#
#   The initial search node is not included.
#
#   A successor that is examined but discarded because its
#   state has already been reached (BFS/DFS), or because it
#   does not improve the best known path cost (UCS/A*), is
#   not counted as generated.
#
#
# max_frontier_size:
#
#   Maximum number of search nodes simultaneously stored
#   in the frontier.
#
#
# path_length:
#
#   Number of actions / graph edges in the returned path.
#
# ============================================================


# ============================================================
# NOT PROVIDED
# TASK 1 — BREADTH-FIRST SEARCH
# ============================================================

def bfs(graph: dict, start: str, end: str) -> str:

    if start not in graph:
        return []

    if end not in graph:
        return []

    queue = deque()
    visited = set()
    parent = {}



    queue.append(start)
    visited.add(start)
    parent[start] = None


    while queue:

        current = queue.popleft()


        if current == end:
            break

        for city in graph[current]:
            if city not in visited:
                visited.add(city)
                parent[city] = current
                queue.append(city)

    if end not in visited:
        return []

    route = []
    current = end

    while current:
        route = [current] + route
        current = parent[current]

    return route








# ============================================================
# NOT PROVIDED
# TASK 2 — DEPTH-FIRST SEARCH
# ============================================================


def dfs(graph: dict, current: str, visited: set, end: str):

    if current == end:
        return [current]

    visited.add(current)


    for neighbour in graph[current]:
        if neighbour not in visited:
            path = dfs(graph, neighbour, visited, end)

            if path:
                return [current] + path

    return None







# ============================================================
# NOT PROVIDED
# TASK 3 — UNIFORM-COST SEARCH
# ============================================================

def ucs(graph:dict, start:str, end:str) -> list:
    priority_queue = [(0,start, [start])]
    visited = set()


    while priority_queue:
        cost, current, path = heapq.heappop(priority_queue)

        if current == end:
            return path, cost


        if current in visited:
            continue

        visited.add(current)


        for neighbour in graph[current]:
            cost = cost + graph[current][neighbour]["distance_km"]

            if neighbour not in visited:
                heapq.heappush(priority_queue, (cost,neighbour,path + [neighbour]))


# ============================================================
# PROVIDED TO STUDENTS
# TASK 4 — GEOGRAPHIC HEURISTIC FORMULA
# ============================================================
#
# Haversine distance:
#
# a =
#   sin²(Δlat / 2)
#   +
#   cos(lat1) cos(lat2) sin²(Δlon / 2)
#
# c =
#   2 asin(sqrt(a))
#
# distance =
#   R × c
#
# R = 6371 km
#
# Angles must be converted from degrees to radians.
#
# ============================================================




# ============================================================
# NOT PROVIDED
# TASK 4 — HAVERSINE IMPLEMENTATION
# ============================================================


def haversine(lat1: float, long1: float, lat2: float, long2:float):
    
    return distance







# ============================================================
# NOT PROVIDED
# TASK 4 — DISTANCE HEURISTIC
# ============================================================


# ============================================================
# PROVIDED TO STUDENTS
# TASK 5 — TIME HEURISTIC IDEA
# ============================================================
#
# Straight-line distance is measured in kilometres.
#
# A lower bound on travel time can be obtained by:
#
#     straight-line distance
#     ----------------------
#     maximum network speed
#
# ============================================================


# ============================================================
# NOT PROVIDED
# TASK 5 — TIME HEURISTIC IMPLEMENTATION
# ============================================================



# ============================================================
# NOT PROVIDED
# TASK 6 — A*
# ============================================================



# ============================================================
# NOT PROVIDED
# INSTRUCTOR TOOL — TRUE OPTIMAL COSTS TO GOAL
# ============================================================
#
# Used to validate heuristics.
#
# This should NOT be part of the student implementation
# because it would reveal most of a UCS / Dijkstra solution.
#
# ============================================================


# ============================================================
# NOT PROVIDED
# INSTRUCTOR TOOL — HEURISTIC ANALYSIS
# ============================================================


    # ========================================================
    # ADMISSIBILITY
    # ========================================================


    # ========================================================
    # CONSISTENCY
    # ========================================================


# ============================================================
# PROVIDED TO STUDENTS
# 19. MAIN SEARCH PROBLEM
# ============================================================

START = normalize_city(
    "Bucharest"
)

GOAL = normalize_city(
    "Rome"
)


assert START in graph, (
    f"Unknown initial state: {START}"
)

assert GOAL in graph, (
    f"Unknown goal state: {GOAL}"
)

assert START != GOAL, (
    "Initial state and goal state must be different."
)


print(
    "\nInitial state:",
    START
)

print(
    "Goal state:",
    GOAL
)


# ============================================================
# NOT PROVIDED
# 20. RUN REFERENCE SEARCH ALGORITHMS
# ============================================================



# ============================================================
# NOT PROVIDED
# 21. VALIDATE ALL RETURNED PATHS
# ============================================================



# ============================================================
# NOT PROVIDED
# 22. HEURISTIC ANALYSIS
# ============================================================


# ============================================================
# PROVIDED TO STUDENTS
# 23. SUMMARY TABLE HELPER
# ============================================================

def make_summary_row(
    name,
    result,
):

    solution = (
        result["solution"]
    )

    diagnostics = (
        result["diagnostics"]
    )


    return {

        "algorithm":
            name,


        "path":
            (
                " -> ".join(
                    solution["path"]
                )

                if solution["path"]

                else None
            ),


        "actions":
            solution[
                "path_length"
            ],


        "distance_km":
            solution[
                "distance_km"
            ],


        "time_min":
            solution[
                "time_min"
            ],


        "expanded":
            diagnostics[
                "expanded_nodes"
            ],


        "generated":
            diagnostics[
                "generated_nodes"
            ],


        "max_frontier":
            diagnostics[
                "max_frontier_size"
            ],
    }


# ============================================================
# NOT PROVIDED
# 24. CREATE COMPARISON TABLE
# ============================================================



# ============================================================
# NOT PROVIDED
# 25. PRINT PATHS AND EXPANSION ORDERS
# ============================================================



# ============================================================
# NOT PROVIDED
# 26. PRINT HEURISTIC ANALYSIS
# ============================================================



# ============================================================
# NOT PROVIDED
# 27. SANITY CHECKS
# ============================================================



# ============================================================
# NOT PROVIDED
# 28. BUILD FINAL RESULTS OBJECT
# ============================================================
#
# Notice:
#
# Search results are divided into:
#
#     solution
#     diagnostics
#
# The solution fields are the strongest candidates for
# strict automatic grading.
#
# Diagnostics are useful for debugging and understanding
# search behaviour.
#
# ============================================================




# ============================================================
# PROVIDED TO STUDENTS
# 29. EXPORT RESULTS.JSON
# ============================================================

OUTPUT_FILE = Path(
    "results.json"
)


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False,
    )


print()

print(
    f"Created: {OUTPUT_FILE}"
)


# ============================================================
# PROVIDED TO STUDENTS
# 30. VERIFY EXPORTED JSON
# ============================================================



"""
with open(
    OUTPUT_FILE,
    "r",
    encoding="utf-8",
) as f:

    loaded_results = (
        json.load(f)
    )


assert (
    loaded_results
    ==
    results
), (
    "results.json does not match the results "
    "generated by the notebook."
)


print(
    "results.json matches the results "
    "generated by the notebook."
)

"""

Number of cities: 20
Number of connections: 35
Dataset loaded successfully.

Canonical city identifiers:
  - athens
  - bari
  - belgrade
  - bratislava
  - bucharest
  - budapest
  - cluj_napoca
  - iasi
  - ljubljana
  - naples
  - plovdiv
  - rome
  - sarajevo
  - skopje
  - sofia
  - thessaloniki
  - timisoara
  - varna
  - vienna
  - zagreb
{'bucharest': {'iasi': {'distance_km': 390.0, 'time_min': 360.0}, 'cluj_napoca': {'distance_km': 445.0, 'time_min': 390.0}, 'timisoara': {'distance_km': 550.0, 'time_min': 450.0}, 'sofia': {'distance_km': 385.0, 'time_min': 330.0}, 'varna': {'distance_km': 270.0, 'time_min': 250.0}, 'belgrade': {'distance_km': 600.0, 'time_min': 540.0}, 'budapest': {'distance_km': 835.0, 'time_min': 510.0}}, 'rome': {'ljubljana': {'distance_km': 755.0, 'time_min': 450.0}, 'naples': {'distance_km': 225.0, 'time_min': 145.0}, 'bari': {'distance_km': 430.0, 'time_min': 285.0}}, 'vienna': {'budapest': {'distance_km': 243.0, 'time_min': 160.0}, 'bratislava': {'dista

NameError: name 'results' is not defined